# Resumiendo

En este notebook aprenderás a resumir texto mediante prompt engineering con foco en topics específicos.

## Setup

In [1]:
import openai
import os
from IPython.display import Markdown
import requests
import json

from dotenv import load_dotenv
load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")
auth_token = os.getenv("AUTH_TOKEN")


headers = {
  'Content-Type': 'application/json',
  'Authorization': f"Bearer {auth_token}"
}

url = 'https://api.awanllm.com/v1/chat/completions' # URL for the API
# Función para llamar a la API de LLMCloud para preguntarle a LLAMA
def preguntar_llama(sys_prompt, prompt):
    payload = json.dumps({
        "model": "Meta-Llama-3-8B-Instruct",
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt}
        ],
      })
    return requests.request("POST", url, headers=headers, data=payload).json()['choices'][0]['message']['content']

client = openai.OpenAI()

def get_completion(prompt, model="gpt-3.5-turbo-0125"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
        max_tokens=512,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )
    return response.choices[0].message.content

## Texto a resumir

In [2]:
text = """
Los agujeros negros, que se derivan de las predicciones de la teoría general de la relatividad de Einstein (1915), tienen sus raíces en las teorías del siglo XVIII. John Michell, un físico visionario, postuló en 1784 la existencia de estrellas con la densidad del Sol, pero de mayor tamaño, dispersas por todo el cosmos. Estas gigantescas estrellas tendrían una gravedad tan intensa que ni siquiera la luz podría escapar de su atracción. Michell estimó que estas estrellas negras tendrían un tamaño aproximado de 500 veces el del Sol, lo que equivaldría a diámetros de alrededor de cien millones de kilómetros.

Hoy en día, gracias a los avances en la astronomía y la física, sabemos que los agujeros negros se encuentran en diversas regiones del universo. Estos objetos cósmicos tienen una masa y un tamaño tan enormes que su gravedad impide que incluso su propia luz escape y, al mismo tiempo, afectan gravitacionalmente a otras estrellas cercanas. Michell anticipó que la influencia gravitacional de estas estrellas negras en otras estrellas podría ser la clave para detectar su existencia.

Además de los agujeros negros estelares, que son el resultado del colapso gravitacional de estrellas masivas, también existen agujeros negros supermasivos, que se encuentran en el centro de muchas galaxias, incluida la nuestra. El agujero negro supermasivo Sagitario A* se ubica en el centro de la Vía Láctea y, aunque no es visible a simple vista, su presencia se evidencia a través del comportamiento peculiar de las estrellas cercanas a él.

La observación de agujeros negros se ha vuelto más sofisticada en las últimas décadas. Por ejemplo, en 2019, el Telescopio del Horizonte de Sucesos capturó la primera imagen de un agujero negro, ubicado en la galaxia M87. Además, las ondas gravitacionales detectadas por LIGO y Virgo han proporcionado información valiosa sobre la fusión de agujeros negros. Estos avances en la investigación de agujeros negros han confirmado muchas de las teorías propuestas por Michell y otros científicos y han contribuido significativamente a nuestra comprensión del universo y sus fenómenos más elusivos.
"""

## Resumir con límite de palabras

In [3]:
prompt = f"""
Tu tarea es generar un breve resumen de un texto científico en palabras llanas y fácilmente entendibles.

Resume el texto que aparece a continuación, delimitado por triple comillas, en menos de 40 palabras. 

Resume: ```{text}```
"""

response = get_completion(prompt)
display(Markdown(f"### Respuesta de GPT\n{response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"### Respuesta de LLAMA\n{preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt sin preámbulos explicativos ni resúmenes finales.', prompt)}"))

### Respuesta de GPT
Los agujeros negros, predichos por Einstein en 1915, se originan en teorías del siglo XVIII. Michell anticipó estrellas gigantes con tanta gravedad que ni la luz puede escapar. Hoy sabemos que existen agujeros negros estelares y supermasivos, como Sagitario A* en la Vía Láctea. Avances recientes en astronomía han confirmado estas teorías y ampliado nuestra comprensión del universo.


---------------------------------------------



### Respuesta de LLAMA
"Los agujeros negros se originaron en la teoría de Einstein y fueron predichos por John Michell en 1784. Ahora se conocen en el universo y se estudian mediante telescopios y detectores de ondas gravitacionales."

In [5]:
# Vamos a contar en cuántas palabras ha hecho el resumen

palabras = response.split()
print(f"Ha resumido el texto en {len(palabras)} palabras")

Ha resumido el texto en 58 palabras


## Resumir haciendo énfasis en un topic específico

In [6]:
prompt = f"""
Tu tarea es generar un breve resumen de un texto científico en palabras llanas y fácilmente entendibles.

Resume el texto que aparece a continuación, delimitado por triple comillas, en menos de 40 palabras y centrándote en las primeras teorías de los agujeros negros. 

Resume: ```{text}```
"""

response = get_completion(prompt)
display(Markdown(f"### Respuesta de GPT\n{response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"### Respuesta de LLAMA\n{preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt sin preámbulos explicativos ni resúmenes finales.', prompt)}"))

### Respuesta de GPT
En el siglo XVIII, John Michell propuso la existencia de estrellas con tanta gravedad que ni la luz podía escapar, los agujeros negros. Hoy sabemos que existen en el universo y su estudio ha avanzado mucho.


---------------------------------------------



### Respuesta de LLAMA
"En 1784, John Michell predijo la existencia de estrellas muy densas y grandes con gravedad tan fuerte que no permitiría que la luz escapara. Esta idea se basaba en la teoría de la relatividad de Einstein y hoy se conocen agujeros negros estelares y supermasivos en el universo."

## Resumir de forma cronológica

In [7]:
prompt = f"""
Tu tarea es generar un breve resumen de un texto científico en palabras llanas y fácilmente entendibles.

Resume el texto que aparece a continuación, delimitado por triple comillas, en menos de 40 palabras y poniendo énfasis en la cronología de los eventos más relevantes.

Resume: ```{text}```
"""

response = get_completion(prompt)
display(Markdown(f"### Respuesta de GPT\n{response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"### Respuesta de LLAMA\n{preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt sin preámbulos explicativos ni resúmenes finales.', prompt)}"))

### Respuesta de GPT
En el siglo XVIII, John Michell predijo la existencia de estrellas negras con gravedad extrema. Hoy sabemos que los agujeros negros existen en el universo, incluyendo agujeros negros estelares y supermasivos en galaxias como la Vía Láctea. Avances recientes, como la primera imagen de un agujero negro capturada en 2019, han confirmado estas teorías y mejorado nuestra comprensión del universo.


---------------------------------------------



### Respuesta de LLAMA
"Años después de que John Michell predijera la existencia de estrellas con gravedad tan intensa que ni la luz podía escapar, se descubrieron agujeros negros en diversas partes del universo. En 2019, se capturó la primera imagen de uno en la galaxia M87, confirmando teorías anteriores."

## Probemos a extraer en vez de resumir

In [8]:
prompt = f"""
Tu tarea es extraer la información relevante del texto que aparece a continuación, delimitado por triple comillas, poniendo énfasis en la cronología de los eventos más relevantes.

Resume: ```{text}```
"""

response = get_completion(prompt)
display(Markdown(f"### Respuesta de GPT\n{response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"### Respuesta de LLAMA\n{preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt sin preámbulos explicativos ni resúmenes finales.', prompt)}"))

### Respuesta de GPT
La cronología de los eventos relevantes sobre los agujeros negros es la siguiente:

- En 1784, John Michell postuló la existencia de estrellas con la densidad del Sol pero de mayor tamaño, que podrían ser tan grandes que ni siquiera la luz podría escapar de su atracción.
- En 1915, la teoría general de la relatividad de Einstein proporcionó las bases teóricas para la existencia de los agujeros negros.
- En la actualidad, se sabe que los agujeros negros se encuentran en diversas regiones del universo, incluidos los agujeros negros estelares y los agujeros negros supermasivos en el centro de las galaxias.
- En 2019, el Telescopio del Horizonte de Sucesos capturó la primera imagen de un agujero negro en la galaxia M87.
- Las ondas gravitacionales detectadas por LIGO y Virgo han proporcionado información valiosa sobre la fusión de agujeros negros, confirmando muchas teorías propuestas por Michell y otros científicos.


---------------------------------------------



### Respuesta de LLAMA
Aquí está la información relevante del texto, enfocada en la cronología de los eventos:

* 1784: John Michell postula la existencia de estrellas con la densidad del Sol, pero de mayor tamaño, que tendrían una gravedad tan intensa que ni siquiera la luz podría escapar de su atracción.
* Siglo XX: La teoría general de la relatividad de Einstein (1915) predice la existencia de agujeros negros.
* Siglo XX: Los avances en la astronomía y la física permiten detectar agujeros negros en diversas regiones del universo.
* 2019: Se captura la primera imagen de un agujero negro, ubicado en la galaxia M87, mediante el Telescopio del Horizonte de Sucesos.
* Siglo XXI: Las ondas gravitacionales detectadas por LIGO y Virgo proporcionan información valiosa sobre la fusión de agujeros negros.

## Resumiendo varios textos a la vez, podemos extraer los puntos en común entre ellos.

In [9]:
text_1 = """
James Clerk Maxwell fue un poeta galardonado, además de físico, y en su ultimo poema Oda paradójica (1878); reflexiona sobre las conexiones entre ciencia, religión y naturaleza, tocando dimensiones superiores en el camino:

Ya que todas las herramientas para mi desatar
en el espacio de cuatro dimensiones yacen,
donde la fantasía juguetona se intercala
avenidas enteras de universos.
"""

text_2 = """
Herbert George Wells.
Con la influencia de ideas científicas, entre ellas, las nuevas geometrías no-euclidianas de Lobachevsky, Riemman y Bolyai, que describían espacios de 4 o más dimensiones, Wells asoció nuestras dimensiones espaciales a las tres primeras coordenadas de un espacio de cuatro dimensiones, y asoció el tiempo a la última dimensión. De esta manera, Wells postuló que los viajes podían realizarse no sólo en el espacio sino que también en el tiempo. La máquina del tiempo (1895) «¿Nunca ha vislumbrado en su conciencia que nada se interponía entre los hombres y una geometría de cuatro dimensiones: largo, ancho, grosor y duración - ¿Pero la inercia de la opinión? ... Cuando tomamos esta nueva luz de una cuarta dimensión y reexaminamos nuestra ciencia física en su iluminación ... ... ya no nos encontramos limitados por una restricción desesperada a un cierto latido del tiempo». O en su obra The Plattner story (1896) que rima con descubrimiento de Möbius de que la reflexión no es sino una rotación en más dimensiones, y la conexión de esto con «el problema de las contrapartes incongruentes» de Kant."""

text_3 = """
El escritor Jorge Luis Borges menciona en un fragmento escrito en 1934 para el diario Crítica titulado "La cuarta dimensión": "Hacia 1670, el plotiniano inglés Henry More usó la frase 'cuarta dimensión', acaso por primera vez en el mundo. No importa lo que quiso comunicar; lo memorable es el contacto genial de esas dos palabras, antes no combinadas. La fórmula intrigó; los hombres no la dejaron morir. Justificar esa conexión de dos términos acaso incompatibles fue, con el tiempo, una de las obligaciones del geómetra. Kant, hacia 1768, estudió ese problema (…) Rehusar la cuarta dimensión es limitar el mundo; afirmarla es enriquecerlo". En una carta a Maurice Abramowicz, en noviembre de 1920, Borges había llegado a escribir: "Como ultraísta y kantiano, creo en la cuarta dimensión". En "There are more things", Borges escribe "los falaces cubos de Hinton o las bien concertadas pesadillas del joven Wells"; y en "Otras inquisiciones" la cuarta dimensión reaparece y el nombre de Kant es mencionado como ejemplo del espíritu platónico que cree en ella. Seguramente Borges no desconocía el "Ensayo" para una nueva cosmogonía, en el que Leopoldo Lugones previene así: "[Y] quizá más pronto de lo que se cree, las especulaciones sobre la cuarta dimensión del espacio puedan darnos un esquema del origen de nuestra geometría".
"""

In [10]:
textos = [text_1, text_2, text_3]

In [11]:
for i in range(len(textos)):
    prompt = f"""
    Tu tarea es generar un breve resumen de un texto.
    
    Resume el texto que aparece a continuación, delimitado por triple comillas, en menos de 40 palabras.

    Textos: ```{textos[i]}```
    """

    response = get_completion(prompt)
    display(Markdown(f"### Respuesta de GPT al texto {i}:\n{response}"))
    display(Markdown(f"### Respuesta de LLAMA al texto {i}:\n{preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt sin preámbulos explicativos ni resúmenes finales.', prompt)}"))
    print("\n---------------------------------------------\n")

### Respuesta de GPT al texto 0:
James Clerk Maxwell, físico y poeta galardonado, reflexiona en su último poema sobre las conexiones entre ciencia, religión y naturaleza, explorando dimensiones superiores.

### Respuesta de LLAMA al texto 0:
"James Clerk Maxwell, premiado poeta y físico, reflexiona sobre conexión entre ciencia, religión y naturaleza en su poema "Oda paradójica", abarcando dimensiones superiores."


---------------------------------------------



### Respuesta de GPT al texto 1:
Herbert George Wells asoció nuestras dimensiones espaciales a un espacio de cuatro dimensiones, incluyendo el tiempo como la cuarta dimensión, lo que permitía viajes no solo en el espacio, sino también en el tiempo.

### Respuesta de LLAMA al texto 1:
"Herbert George Wells desarrolló la teoría de la máquina del tiempo, inspirado en geometrías no-euclidianas, considerando el tiempo como la cuarta dimensión y permitiendo viajar en el tiempo."


---------------------------------------------



### Respuesta de GPT al texto 2:
En el texto, Borges menciona la importancia de la cuarta dimensión en la filosofía y la literatura, haciendo referencia a Kant y otros autores que exploraron este concepto.

### Respuesta de LLAMA al texto 2:
"Jorge Luis Borges habla sobre la cuarta dimensión, citando a Henry More y Kant, y expresa su creencia en ella como ultraísta y kantiano."


---------------------------------------------



In [12]:
prompt = f"""
Tu tarea es generar un breve resumen de unos textos.

Resume los textos que aparecen a continuación, delimitados por triple comillas, en menos de 40 palabras.

Extrae además el tema o temas en común de los textos. Formatea tu respuesta apropiadamente.

Textos: ```{text_1} {text_2} {text_3}```
"""

response = get_completion(prompt)
display(Markdown(f"### Respuesta de GPT\n{response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"### Respuesta de LLAMA\n{preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt sin preámbulos explicativos ni resúmenes finales.', prompt)}"))

### Respuesta de GPT
Los textos abordan la relación entre la ciencia y la cuarta dimensión, explorando las conexiones entre la física, la geometría y la filosofía. Autores como Maxwell, Wells y Borges reflexionan sobre la expansión del conocimiento más allá de las tres dimensiones tradicionales.


---------------------------------------------



### Respuesta de LLAMA
**Resumen**: Los textos hablan sobre la idea de la cuarta dimensión, que se aborda desde diferentes perspectivas científicas y literarias. Se citan ejemplos de autores como James Clerk Maxwell, Herbert George Wells y Jorge Luis Borges, que exploran conceptos como la relatividad y la geometría no-euclidea.

**Tema en común**: La cuarta dimensión como tema transversal en la ciencia y la literatura.

Cómo podemos ver, el resumen de texto es una tarea muy versátil y puede ser adaptada a diferentes necesidades. Además de resumir texto, también podemos extraer información relevante de los textos solo con cambiar un poco el prompt. Un caso de uso habitual es resumir opiniones de clientes sobre un producto, para poder identificar los puntos en común entre las opiniones y mejorar el producto, haciendo énfasis en los puntos más mencionados por los clientes. 

## Tu turno de experimentar!
